Coding Question 1

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

data = {
    "area_sqft": [750, 900, 1200, 1500, 1800, 2100, 2400, 2700, 3000, 3500],
    "distance_school_km": [1.2, 2.1, 1.0, 3.5, 2.8, 4.0, 3.0, 5.2, 4.5, 6.0],
    "house_price": [45, 52, 65, 72, 85, 95, 110, 125, 140, 165]
}

df = pd.DataFrame(data)

# task 1 - separate features and target
X = df[["area_sqft", "distance_school_km"]]
y = df["house_price"]

# task 2 - create pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor())
])

# task 3 & 4 - gridsearchcv with 5-fold cv
param_grid = {"model__n_neighbors": [1, 2, 3, 4, 5]}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="r2")
grid.fit(X, y)

# task 5 - best parameter and score
print("Best Parameter:", grid.best_params_)
print("Best Score:", round(grid.best_score_, 4))

# task 6 - cv results as dataframe
results = pd.DataFrame(grid.cv_results_)
print(results[["param_model__n_neighbors", "mean_test_score", "rank_test_score"]])

Best Parameter: {'model__n_neighbors': 1}
Best Score: -12.5449
   param_model__n_neighbors  mean_test_score  rank_test_score
0                         1       -12.544907                1
1                         2       -16.967846                2
2                         3       -19.184854                3
3                         4       -23.987305                4
4                         5       -29.279427                5


Coding Question 2

In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_score, recall_score, confusion_matrix, make_scorer

data = {
    "transaction_amount": [200, 4500, 120, 8000, 90, 15000, 300, 700, 20000, 150],
    "transaction_time": [10, 2, 14, 3, 16, 1, 11, 13, 2, 17],
    "previous_failed_attempts": [0, 3, 0, 4, 0, 5, 1, 0, 6, 0],
    "fraud": [0, 1, 0, 1, 0, 1, 0, 0, 1, 0]
}

df = pd.DataFrame(data)

# task 1 - split into X and y
X = df[["transaction_amount", "transaction_time", "previous_failed_attempts"]]
y = df["fraud"]

# task 2 - pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
])

# task 3 - custom scorer using minimum of precision and recall
def min_precision_recall(y_true, y_pred):
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    return min(p, r)

custom_scorer = make_scorer(min_precision_recall)

# task 4 - gridsearchcv
param_grid = {"model__C": [0.01, 0.1, 1, 10]}

grid = GridSearchCV(pipe, param_grid, scoring=custom_scorer, cv=3)
grid.fit(X, y)

# task 5 - best C value
print("Best C:", grid.best_params_)

# task 6 - evaluate final model
y_pred = grid.best_estimator_.predict(X)

print("Precision:", round(precision_score(y, y_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y, y_pred, zero_division=0), 4))
print("Confusion Matrix:\n", confusion_matrix(y, y_pred))

Best C: {'model__C': 0.01}
Precision: 1.0
Recall   : 1.0
Confusion Matrix:
 [[6 0]
 [0 4]]


Coding Question 3

In [3]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

data = {
    "study_hours": [1, 2, 3, 4, 5, 6, 7, 8],
    "attendance": [40, 50, 55, 60, 70, 75, 85, 90],
    "learning_mode": ["online", "offline", "online", "offline",
                      "hybrid", "hybrid", "offline", "online"],
    "passed": [0, 0, 0, 1, 1, 1, 1, 1]
}

df = pd.DataFrame(data)

# task 1 - separate numerical and categorical columns
numerical_cols = ["study_hours", "attendance"]
categorical_cols = ["learning_mode"]

X = df[numerical_cols + categorical_cols]
y = df["passed"]

# task 2, 3, 4 - columntransformer with polynomial and onehot
preprocessor = ColumnTransformer([
    ("poly", PolynomialFeatures(degree=2, include_bias=False), numerical_cols),
    ("ohe", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

# task 5 - full pipeline with logistic regression
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

pipe.fit(X, y)

# task 6 - predict new student
new_student = pd.DataFrame({
    "study_hours": [5],
    "attendance": [72],
    "learning_mode": ["online"]
})

prediction = pipe.predict(new_student)
print("Prediction (0 = Fail, 1 = Pass):", prediction[0])

Prediction (0 = Fail, 1 = Pass): 1
